# Градиентный спуск

## 1. Базовые элементы обучения

1. **Ошибка** — показывает, как сильно мы не правы.
2. **Градиент** — указывает направление, куда менять параметры.
3. **Способ оценки градиента (GD, SGD, Mini-batch, SAG)** — решает, по какой части данных мы этот градиент считаем (экономим ресурсы).
4. **Оптимизатор (Momentum, RMSProp, Adam)** — решает, с какой физикой и динамикой (инерция, адаптивная скорость) мы делаем сам шаг в этом направлении.
   
5. **Признаки ($X$)** — это то, что подается на вход. Они неизменны во время обучения.
6. **Веса ($w$)** — память и интеллект модели. Обучить модель — значит найти правильные числовые значения для ее весов. Определяют, насколько сильно каждый признак важен для итоговой цены.
7. **Предсказание** — E(X * W) результат умножения всех признаков на текущие веса.
8. **Функционал ошибки ($Q$)** — сравнивает то, что предсказала модель, с реальностью, выдает одно число — насколько сильно модель облажалась.
9.  **Разреженный признак** — признак, который почти на всех объектах равен нулю.

В машинном обучении «самый низ» означает, что наша модель наилучшим образом выучила закономерности из данных. Давайте разберем, что это значит на практике:

Идеальные предсказания: Если вы находитесь в самом низу горы (ошибка равна нулю или минимальна), это значит, что ваши предсказания (например, стоимость квартиры или диагноз) максимально близки к реальным правильным ответам (ground truth).

Максимальная точность для бизнеса/задачи: Модель, которая находится на дне «оврага» ошибки, работает стабильно и предсказуемо. Если же остановиться где-то на полпути (на склоне), модель будет постоянно промахиваться и выдавать неточные результаты.

## Зачем мы считаем «градиент ошибки»?
- Мы считаем градиент ошибки, чтобы ответить на единственный вопрос: «В какую сторону крутить веса (параметры) модели, чтобы ошибка уменьшилась?».
- Если у вашей модели есть 1000 параметров (весов), и она делает неверные предсказания, вы не знаете, какой именно вес виноват и насколько его изменить.
- Градиент — это математический инструмент, который рассчитывает для каждого из 1000 весов его личный вклад в общую ошибку. Он говорит: «Вес №1 нужно увеличить чуть-чуть, вес №2 уменьшить сильно, а вес №3 вообще не трогать».

## 2. Способы оценки градиента (посчитать ошибку)
«какую часть данных взять, чтобы посчитать направление следующего шага?»
* **Full-Batch Gradient Descent (GD)** Вычисляет ошибку и градиент по всей обучающей выборке на каждом шаге ($N$ объектов) перед тем, как сделать один единственный шаг изменения весов.
  - Плюсы: Идеально плавный и точный спуск к минимуму без хаотичных колебаний.
  - Минусы: На огромных датасетах один шаг требует колоссальных вычислений и не помещается в оперативной памяти (RAM/VRAM).
* **SGD (Стохастический градиентный спуск):** На каждом шаге берет ровно один случайный объект (или пару объектов), считает градиент только по нему и сразу обновляет веса.
  - Плюсы: Мгновенный пересчет весов, возможность обучать модели на данных, которые не влезают в память (потоковая обработка), а также способность «выпрыгивать» из мелких локальных минимумов за счет хаотичного шума.
  - Минусы: Траектория движения крайне нестабильна («шумит»), из-за чего модель может долго блуждать вокруг минимума, а при сильном шуме или сильной разреженности данных сходимость нарушается.
* **Mini-Batch Gradient Descent:** Компромиссный и самый распространенный в индустрии вариант. Данные делятся на небольшие пакеты (батчи) фиксированного размера $n$ (например, 32, 64, 128 или 256 объектов). Градиент считается усредненно по одному такому батчу.
  - Плюсы: Объединяет достоинства обоих методов — работает достаточно быстро (за счет векторизации вычислений на CPU/GPU) и при этом дает стабильную, сглаженную траекторию спуска.
* **Stochastic Average Gradient (SAG) и ее развитие (SAGA):** Вместо пересчета полного градиента по всей базе или использования чисто случайного объекта, алгоритм хранит в памяти последние вычисленные градиенты для каждого отдельного объекта выборки. На каждом шаге обновляется градиент только для одного случайно выбранного объекта, а для остальных берутся старые значения из памяти.
  - Плюсы: Позволяет добиться высокой точности сходимости (как у полного GD), сохраняя скорость итераций на уровне SGD.
  - Минусы: Требует дополнительной памяти для хранения таблицы градиентов для каждого объекта датасета ($O(N)$), что становится критично при огромных размерах баз данных.

## 3. Общее количество шагов - это гибкая настройка (гиперпараметр)
«Сколько шагов в одной эпохе (один проход по всему миллиону данных)?»

* **Full-Batch Gradient Descent (GD)** - 1 очень долгий шаг
* **SGD (Стохастический градиентный спуск)** - 1 000 000 шагов
* **Mini-Batch Gradient Descent** ~ 15 625 шагов (батч по 64)
* **Stochastic Average Gradient (SAG) и ее развитие (SAGA)** - 1 000 000 шагов но за счет таблицы памяти - качественнее
  
Инженер сам решает, какого размера делать пачки и сколько раз крутить данные, исходя из мощности компьютера и сложности задачи.

## 4. Оптимизаторы
«А как именно сделать этот шаг по найденному направлению?»

Это умные надстройки над градиентным спуском, которые «на ходу» подстраивают размер и направление каждого шага под рельеф местности.

* **Momentum (импульс):** учитывает не только текущий наклон (градиент), но и инерцию от предыдущих шагов. (Тяжелый металлический шар)
* **AdaGrad:** индивидуально настраивает скорость шага (Learning Rate) для каждого параметра модели.
  - Главный минус: Он делит шаги на сумму всех прошлых квадратов градиентов. Из-за этого к середине обучения эта сумма становится огромной, и скорость шага (Learning Rate) падает практически до нуля. Модель «засыпает» и перестает учиться.
* **RMSProp:** Исправляет баг AdaGrad. Вместо того чтобы сумплировать вообще все прошлые градиенты за всю историю обучения, он использует экспоненциально скользящее среднее (помнит только недавнее прошлое).
* **Adam (Adaptive Moment Estimation):** Это идеальный гибрид. Он объединяет лучшее от Momentum (инерция шара, чтобы быстро катиться и гасить колебания) и от RMSProp (автоматическая адаптация скорости под каждый параметр).
  - Почему его все любят: Adam работает «из коробки» почти в любой задаче (от простых таблиц до нейросетей для генерации картинок и текста). Он сам понимает, где нужно ускориться, где затормозить, и избавляет инженера от мучительного подбора параметров.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Простая демонстрация градиентного спуска для функции потерь Q(w) = w^2
def loss_function(w):
    return w ** 2

def gradient(w):
    return 2 * w

w = 10.0  # Начальное значение веса
learning_rate = 0.1
history = [w]

# Шаги градиентного спуска
for _ in range(10):
    grad = gradient(w)
    w = w - learning_rate * grad
    history.append(w)

print("Эволюция веса в процессе градиентного спуска:", [round(val, 4) for val in history])

## 5. Расчет ошибок
### А. Для задач регрессии (предсказание чисел, цен, температуры)
#### 1. MSE (Mean Squared Error / Среднеквадратическая ошибка)
Расчет функционала ошибки в задачах регрессии
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
- Где: $y_i$ — реальность, $\hat{y}_i$ — прогноз модели, $n$ — количество объектов в пачке.
- Особенность: Возводит ошибки в квадрат. Сильно наказывает модель за грубые промахи.
#### 2. MAE (Mean Absolute Error) — Средняя абсолютная ошибка
$$MAE = \frac{1}{n} \sum_{i=1}^{n} \vert{}y_i - \hat{y}_i\vert{}$$
- Особенность: Просто модуль разницы (без квадратов). Все ошибки весят одинаково. Если в данных много случайных выбросов (например, странных цен), MAE справляется лучше, чем MSE, потому что не «бесится» от гигантских квадратов.
### Б. Для задач классификации (предсказание категорий: кот/собака, спам/не спам)
1. Binary Cross-Entropy (Бинарная кросс-энтропия)
   - Применяется, когда классов всего два (0 или 1, да или нет). Она измеряет «неуверенность» модели: если объект был спамом ($y = 1$), а модель была уверена на 99%, ошибка будет нулевой. Если модель ошиблась и была уверена в обратном, ошибка улетит в бесконечность.
2. Categorical Cross-Entropy (Категориальная кросс-энтропия)
   - Используется, когда классов больше двух (например, классификация одежды: футболка, куртка, кроссовки).

## 6. Переобучение (overfitting)
Train — это наш учебник с ответами на задней странице. Модель решает задачу, сверяется с ответами в train, получает по шапке за ошибки и исправляет свои веса.

Test — это экзамен. Мы прячем его от модели до самого конца. Когда обучение полностью завершено, мы даем модели тест (дома, которые она никогда раньше не видела), чтобы честно проверить, чему она научилась. Если обучать по test, модель просто «зазубрит ответы» (переобучится), а на новых реальных данных облажается.

## 7. Процесс Обучения
1. Взяли пачку (например, 64 дома из train).
2. Посчитали для них предсказания ($\hat{y}$) с помощью текущих весов ($W$).
3. Сравнили прогноз с реальностью ($y$) с помощью формулы MSE, получив числовое значение ошибки.
4. Посчитали градиент (куда и насколько сильно надо изменить веса, чтобы ошибка уменьшилась).
5. Применили оптимизатор (Adam, Momentum и т.д.), который с учетом инерции и адаптивной скорости делает один шаг и обновляет веса ($W$).
6. Переходим к следующей пачке
7. ...

В конце обучения: Получаем фиксированный набор оптимальных весов ($W$) - **обученную модель**